# DQL: Joins

Joins combine rows from two or more tables. This notebook joins employees to departments, projects, and salary grades.


## Setup

Run this first. It creates a Spark session, finds the CSV files, loads them as DataFrames, and registers SQL temp views.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd(),
]

DATA_DIR = next((path for path in candidate_dirs if (path / "emp.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find emp.csv. Put the CSV files in ./data or beside this notebook.")

print(f"Reading CSV files from: {DATA_DIR}")

emp = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp.csv"))
dept = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "dept.csv"))
salgrade = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "salgrade.csv"))
projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "projects.csv"))
emp_projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp_projects.csv"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## Inner Join

An inner join keeps only rows that match in both DataFrames.


In [ ]:
emp.join(dept, on="deptno", how="inner").select("empno", "ename", "job", "dname", "loc").show(10)

spark.sql("""
SELECT e.empno, e.ename, e.job, d.dname, d.loc
FROM emp e
INNER JOIN dept d ON e.deptno = d.deptno
ORDER BY e.empno
LIMIT 10
""").show()


## Left Outer Join

A left join keeps all rows from the left DataFrame and fills unmatched right-side columns with null.


In [ ]:
dept.join(emp, on="deptno", how="left").select("deptno", "dname", "empno", "ename").orderBy("deptno", "empno").show(20)

spark.sql("""
SELECT d.deptno, d.dname, e.empno, e.ename
FROM dept d
LEFT OUTER JOIN emp e ON d.deptno = e.deptno
ORDER BY d.deptno, e.empno
""").show(20)


## Right Outer Join

A right join keeps all rows from the right DataFrame. It is often equivalent to swapping table order and using a left join.


In [ ]:
emp.join(dept, on="deptno", how="right").select("deptno", "dname", "empno", "ename").orderBy("deptno", "empno").show(20)

spark.sql("""
SELECT d.deptno, d.dname, e.empno, e.ename
FROM emp e
RIGHT OUTER JOIN dept d ON e.deptno = d.deptno
ORDER BY d.deptno, e.empno
""").show(20)


## Full Outer Join

A full outer join keeps all rows from both sides. Unmatched columns become null.


In [ ]:
emp.join(dept, on="deptno", how="full").select("deptno", "dname", "empno", "ename").orderBy("deptno", "empno").show(20)

spark.sql("""
SELECT COALESCE(e.deptno, d.deptno) AS deptno, d.dname, e.empno, e.ename
FROM emp e
FULL OUTER JOIN dept d ON e.deptno = d.deptno
ORDER BY deptno, e.empno
""").show(20)


## Joining Multiple Tables

This example connects employees to projects through `emp_projects`.


In [ ]:
emp.join(emp_projects, "empno", "inner").join(projects, "projectno", "inner").select(
    "empno", "ename", "project_name", "budget", "start_date", "end_date"
).orderBy("empno").show(15)

spark.sql("""
SELECT e.empno, e.ename, p.project_name, p.budget, ep.start_date, ep.end_date
FROM emp e
JOIN emp_projects ep ON e.empno = ep.empno
JOIN projects p ON ep.projectno = p.projectno
ORDER BY e.empno
LIMIT 15
""").show()


## Non-Equi Join

Not every join uses equality. Salary grades are matched by checking whether salary falls between `losal` and `hisal`.


In [ ]:
emp.join(salgrade, (emp.sal >= salgrade.losal) & (emp.sal <= salgrade.hisal), "inner").select(
    "empno", "ename", "sal", "grade"
).orderBy("empno").show(15)

spark.sql("""
SELECT e.empno, e.ename, e.sal, s.grade
FROM emp e
JOIN salgrade s ON e.sal BETWEEN s.losal AND s.hisal
ORDER BY e.empno
LIMIT 15
""").show()
